Install packages



In [ ]:
!pip install -q datasets pandas scikit-learn numpy

Importing Libraries

In [ ]:
import re
import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Loading Dataset

The real Hugging Face dataset uses 'movie title -year' and 'rating', so i rename them to 'title and 'rating' for consistency.

In [ ]:
train_ds = load_dataset('jquigl/imdb-genres', split='train')
val_ds = load_dataset('jquigl/imdb-genres', split='validation')

train_df = train_ds.to_pandas()
val_df = val_ds.to_pandas()

train_df = train_df.rename(columns={'movie title - year': 'title', 'rating': 'ratings'})
val_df = val_df.rename(columns={'movie title - year': 'title', 'rating': 'ratings'})

train_df.head()

Keep Required Columns

In [ ]:
required_columns = ['title', 'description']
optional_columns = ['genre', 'expanded-genres', 'ratings']

for col in optional_columns:
    if col not in train_df.columns:
        train_df[col] = ''
    if col not in val_df.columns:
        val_df[col] = ''

train_df = train_df[required_columns + optional_columns].copy()
val_df = val_df[required_columns + optional_columns].copy()

train_df['title'] = train_df['title'].fillna('').astype(str).str.strip()
train_df['description'] = train_df['description'].fillna('').astype(str).str.strip()
train_df['genre'] = train_df['genre'].fillna('').astype(str)
train_df['expanded-genres'] = train_df['expanded-genres'].fillna('').astype(str)

val_df['title'] = val_df['title'].fillna('').astype(str).str.strip()
val_df['description'] = val_df['description'].fillna('').astype(str).str.strip()
val_df['genre'] = val_df['genre'].fillna('').astype(str)
val_df['expanded-genres'] = val_df['expanded-genres'].fillna('').astype(str)

train_df = train_df[(train_df['title'] != '') & (train_df['description'] != '')].reset_index(drop=True)
val_df = val_df[(val_df['title'] != '') & (val_df['description'] != '')].reset_index(drop=True)

print('Train shape:', train_df.shape)
print('Validation shape:', val_df.shape)

Text Preprocessing

applying:
- lowercase
- punctuation removal
- stopword removal

In [ ]:
STOPWORDS = {
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 'and',
    'any', 'are', 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below',
    'between', 'both', 'but', 'by', 'can', 'could', 'did', 'do', 'does', 'doing',
    'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'has', 'have',
    'having', 'he', 'her', 'here', 'hers', 'herself', 'him', 'himself', 'his', 'how',
    'i', 'if', 'in', 'into', 'is', 'it', 'its', 'itself', 'just', 'me', 'more',
    'most', 'my', 'myself', 'no', 'nor', 'not', 'now', 'of', 'off', 'on', 'once',
    'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 's',
    'same', 'she', 'should', 'so', 'some', 'such', 't', 'than', 'that', 'the', 'their',
    'theirs', 'them', 'themselves', 'then', 'there', 'these', 'they', 'this', 'those',
    'through', 'to', 'too', 'under', 'until', 'up', 'very', 'was', 'we', 'were',
    'what', 'when', 'where', 'which', 'while', 'who', 'whom', 'why', 'will', 'with',
    'you', 'your', 'yours', 'yourself', 'yourselves'
}

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = [token for token in text.split() if token not in STOPWORDS]
    return ' '.join(tokens)

train_df['clean_description'] = train_df['description'].apply(clean_text)
val_df['clean_description'] = val_df['description'].apply(clean_text)

train_df[['title', 'clean_description']].head()

Vectorization

TF-IDF


In [ ]:
vectorizer_type = 'tfidf'  # change to 'bow' if needed

if vectorizer_type == 'tfidf':
    vectorizer = TfidfVectorizer(max_features=7000, ngram_range=(1, 2))
else:
    vectorizer = CountVectorizer(max_features=7000, ngram_range=(1, 2))

train_matrix = vectorizer.fit_transform(train_df['clean_description'])
val_matrix = vectorizer.transform(val_df['clean_description'])

print('Train matrix shape:', train_matrix.shape)
print('Validation matrix shape:', val_matrix.shape)

Recommendation Functions

In [ ]:
train_df['title_key'] = train_df['title'].str.lower().str.strip()

def build_query_text(title=None, vague_description=None):
    parts = []
    title_index = None

    if title:
        title_key = title.lower().strip()
        matches = train_df.index[train_df['title_key'] == title_key].tolist()
        if not matches:
            raise ValueError(f'Movie title not found: {title}')
        title_index = matches[0]
        parts.append(train_df.loc[title_index, 'clean_description'])

    if vague_description:
        parts.append(clean_text(vague_description))

    if not parts:
        raise ValueError('Please provide a movie title or a vague description.')

    return ' '.join(parts), title_index

def recommend_movies(title=None, genre=None, vague_description=None, top_k=5):
    query_text, title_index = build_query_text(title=title, vague_description=vague_description)
    query_vector = vectorizer.transform([query_text])
    similarity_scores = cosine_similarity(query_vector, train_matrix).flatten()

    results = train_df.copy()
    results['similarity_score'] = similarity_scores

    if title_index is not None:
        results = results.drop(index=title_index)

    if genre:
        genre_key = genre.lower().strip()
        genre_mask = (
            results['genre'].str.lower().str.contains(genre_key, na=False) |
            results['expanded-genres'].str.lower().str.contains(genre_key, na=False)
        )
        filtered = results[genre_mask]
        if not filtered.empty:
            results = filtered

    columns = ['title', 'genre', 'expanded-genres', 'ratings', 'similarity_score', 'description']
    return results.sort_values('similarity_score', ascending=False).head(top_k)[columns].reset_index(drop=True)

Example Recommendation by Title

In [ ]:

example_title = 'Inception - 2010'

recommend_movies(title=example_title, top_k=5)


Example Recommendation by Vague Description

In [ ]:
recommend_movies(
    vague_description='space survival mission with astronauts',
    genre='Sci-Fi',
    top_k=5
)

Limitations

- only uses the description field
- may rank movies high because of keyword overlap rather than true semantic similarity
- genre filtering is simple text matching
- no collaborative filtering or user behavior signals

Possible Improvements

- compare TF-IDF and Bag-of-Words more systematically
- add stemming or lemmatization
- combine description with metadata such as cast, director, year, or keywords
